# DIF report walkthrough

Simulate data with an embedded DIF effect, fit the Rasch model, run a Lord's chi-square DIF analysis, and produce a grouped HTML validity report.

In [ ]:
import numpy as np

from rasch_per import DIFAnalysis, RaschModel, ResponseData, generate_report
from rasch_per.simulate import simulate_rasch_data

In [ ]:
rng = np.random.default_rng(7)
n = 600
groups = np.where(rng.uniform(size=n) < 0.5, "Man", "Non-man")
# Items 2, 7, 11 are 0.6 logits harder for the Non-man (focal) group
dif_effects = {2: 0.6, 7: 0.6, 11: 0.6}

sim = simulate_rasch_data(
    n_persons=n,
    n_items=20,
    seed=7,
    groups=groups,
    focal_label="Non-man",
    dif_effects=dif_effects,
)
df = sim.set_index("person_id") if "person_id" in sim.columns else sim

In [ ]:
data = ResponseData(df)
model = RaschModel().fit(data, estimator="MML")
dif = DIFAnalysis(
    model,
    groups=groups,
    reference="Man",
    focal="Non-man",
).analyze()
dif.summary()

In [ ]:
generate_report(
    df,
    output="dif_report.html",
    groups=groups,
    reference="Man",
    focal="Non-man",
)
print("Wrote dif_report.html")

## Reading the ETS classification

The `ets_class` column uses the ETS delta scale: **C** (< 1.0 logits, negligible), **B** (1.0 to 1.5, moderate), **A** (> 1.5, large). Items flagged by Benjamini-Hochberg (`flag = True`) after FDR control are the ones to scrutinize for bias.